# GPT: trening, skala i analiza zachowania

## Zadanie
Zbadaj GPT jako architekturę decoder-only i procedurę treningową. Eksperyment ilustruje rolę maski causal, oszacujemy parametry i koszt attention, a następnie porównamy GPT-3, GPT-4 i ChatGPT. Szczegóły GPT-4 nie są publiczne w stopniu pozwalającym odtworzyć jego liczbę parametrów.

## 1. Autoregresja i przeciek przyszłości
GPT modeluje $p(x_1,…,x_T)=∏_t p(x_t|x_{<t})$. Pretraining minimalizuje ujemny log-likelihood następnego tokenu. Bez maski model mógłby odczytać przyszłe tokeny. Poniższy eksperyment syntetyczny ilustruje problem; unmasked attention jest właściwe dla niektórych encoderów, ale nie dla GPT.

In [ ]:
import math,torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(4);device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
class TinyLM(nn.Module):
    def __init__(self,causal=True):
        super().__init__();self.causal=causal;self.e=nn.Embedding(16,32);self.qkv=nn.Linear(32,96);self.o=nn.Linear(32,16)
    def forward(self,x):
        B,T=x.shape;q,k,v=self.qkv(self.e(x)).chunk(3,-1);s=q@k.transpose(-2,-1)/math.sqrt(32)
        if self.causal:s=s.masked_fill(~torch.tril(torch.ones(T,T,dtype=torch.bool,device=x.device)),torch.finfo(s.dtype).min)
        return self.o(s.softmax(-1)@v)
seq=torch.tensor([[i%4 for i in range(20)] for _ in range(32)],device=device);x,y=seq[:,:-1],seq[:,1:];hist={}
for name,causal in [("causal",True),("bez maski",False)]:
    model=TinyLM(causal).to(device);opt=torch.optim.AdamW(model.parameters(),lr=.01);ls=[]
    for _ in range(100):
        z=model(x);loss=F.cross_entropy(z.reshape(-1,16),y.reshape(-1));opt.zero_grad();loss.backward();opt.step();ls.append(loss.item())
    hist[name]=ls
for n,v in hist.items():plt.plot(v,label=n)
plt.yscale("log");plt.xlabel("krok");plt.ylabel("cross entropy");plt.legend();plt.grid();plt.show()

## 2. Parametry i koszt
Dla $d=d_{model}$: QKV i wyjście attention to około $4d²$, MLP $d→4d→d$ to $8d²$, razem około $12d²$ parametrów na blok (plus biasy/normy). Wagi attention wymagają $O(T²)$ pamięci, a obliczenia $O(T²d)$. Podwojenie kontekstu zwiększa koszt tej części około czterokrotnie.

In [ ]:
def block_params(d):return 12*d*d+13*d
for d in (64,256,768,2048):print(d,f"~{block_params(d):,} parametrów/blok")
T=[128,256,512,1024,2048,4096]
plt.loglog(T,[(t/128)**2 for t in T],marker="o");plt.xlabel("długość T");plt.ylabel("koszt względny (T=128)");plt.title("Kwadratowa złożoność attention");plt.grid(True,which="both");plt.show()

## 3. Od bazowego GPT do asystenta
Typowy schemat (różni się między systemami): pretraining next-token na korpusie; supervised fine-tuning (SFT) na przykładach instrukcja–odpowiedź; dostrajanie preferencji, np. RLHF (model nagrody + optymalizacja polityki) lub DPO; integracja produktu z promptami, narzędziami, routingiem i filtrami. Pretraining nie gwarantuje posłuszeństwa instrukcjom, prawdziwości ani bezpieczeństwa.

## 4. GPT-3, GPT-4, ChatGPT

| Nazwa | Znaczenie | Co wiadomo / ograniczenia |
|---|---|---|
| GPT-3 | Model językowy opisany w 2020 r. | Publikacja opisała wariant do 175 mld parametrów i zdolności few-shot. |
| GPT-4 | Późniejszy model ogólnego przeznaczenia | Raport opisuje ewaluacje, lecz nie ujawnia wielu szczegółów architektury, danych i rozmiaru. |
| ChatGPT | Asystent konwersacyjny | Produkt/rodzina systemów dialogowych, nie synonim jednej architektury lub jednej wagi modelu. |

Nie przenoś liczby parametrów GPT-3 na GPT-4. Nazwa ChatGPT obejmuje zmieniające się systemy; ta część nie jest dokumentacją aktualnego API.

## 5. Analiza modelu i ćwiczenie
Analizuj poziom tokenu (logity, entropia, temperatura), sekwencji (loss, perplexity), zadania (jakość, kalibracja, odporność, wycieki) i systemu (opóźnienie, przepustowość, koszt, kontekst). Perplexity mierzy modelowanie tekstu, ale nie mierzy wprost faktograficzności, użyteczności dialogu ani bezpieczeństwa.

**Zadanie:** przygotuj kartę eksperymentu mini-GPT z poprzedniego notebooka: rozmiar danych, tokenizer, parametry, train/validation loss i PPL, dekodowanie oraz trzy generacje. Opisz ograniczenie, którego nie naprawi sama temperatura.

## Podsumowanie
GPT autoregresyjnie przewiduje tokeny. Maska causal blokuje przeciek, a standardowe attention rośnie kwadratowo względem długości kontekstu. Asystenci dodają dostrajanie oraz elementy systemowe.